In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv(
    "../data/processed/lifetwin_clean_dataset.csv"
)

print("Dataset shape:", df.shape)

Dataset shape: (10000, 20)


In [3]:
regressor = joblib.load(
    "../models/regression/best_regressor.pkl"
)

scaler = joblib.load(
    "../models/preprocessing/scaler.pkl"
)

encoder = joblib.load(
    "../models/preprocessing/encoder.pkl"
)

print("Regressor loaded:", regressor is not None)
print("Scaler loaded:", scaler is not None)
print("Encoder loaded:", encoder is not None)

Regressor loaded: True
Scaler loaded: True
Encoder loaded: True


In [4]:
behavioral_features = [
    "sleep_hours",
    "sleep_quality",
    "wake_time_consistency",
    "study_work_hours",
    "focus_sessions",
    "screen_time_hours",
    "social_media_hours",
    "notification_interruptions",
    "exercise_minutes",
    "step_count",
    "stress_level",
    "mood_score",
    "energy_level",
    "caffeine_intake",
    "break_minutes"
]

categorical_features = [
    "age_group",
    "day_of_week"
]

target = "productivity_score"

print("Behavioral features:", len(behavioral_features))
print("Categorical features:", len(categorical_features))

Behavioral features: 15
Categorical features: 2


In [5]:
def prepare_model_input(row):
    """
    Convert one behavioral record into the exact
    25-feature representation expected by the trained regressor.
    """

    behavioral_data = pd.DataFrame(
        [row[behavioral_features]]
    )

    categorical_data = pd.DataFrame(
        [row[categorical_features]]
    )

    # Scale behavioral features
    behavioral_scaled = scaler.transform(
        behavioral_data
    )

    # Encode categorical features
    categorical_encoded = encoder.transform(
        categorical_data
    )

    # Combine exactly as done during model training
    model_input = np.hstack([
        behavioral_scaled,
        categorical_encoded
    ])

    return model_input

In [6]:
def what_if_prediction(
    record_index,
    changes
):
    """
    Compare baseline productivity with a
    model-estimated What-If scenario.

    changes:
        Dictionary containing behavioral features
        to modify.
    """

    # Original day
    baseline_row = df.loc[record_index].copy()

    # Create scenario from original day
    scenario_row = baseline_row.copy()

    # Apply user-defined changes
    for feature, value in changes.items():

        if feature not in behavioral_features:
            raise ValueError(
                f"{feature} is not a valid behavioral feature."
            )

        scenario_row[feature] = value

    # Prepare model inputs
    baseline_input = prepare_model_input(
        baseline_row
    )

    scenario_input = prepare_model_input(
        scenario_row
    )

    # Predictions
    baseline_prediction = regressor.predict(
        baseline_input
    )[0]

    scenario_prediction = regressor.predict(
        scenario_input
    )[0]

    # Difference
    change = (
        scenario_prediction -
        baseline_prediction
    )

    return {
        "record_index": record_index,
        "baseline_prediction": round(
            float(baseline_prediction), 2
        ),
        "scenario_prediction": round(
            float(scenario_prediction), 2
        ),
        "estimated_change": round(
            float(change), 2
        ),
        "changes": changes
    }

In [7]:
record_index = 0

print(
    df.loc[
        record_index,
        behavioral_features
    ]
)

sleep_hours                      3.00
sleep_quality                    4.50
wake_time_consistency            5.40
study_work_hours                10.58
focus_sessions                   0.90
screen_time_hours                9.88
social_media_hours               4.48
notification_interruptions      12.00
exercise_minutes                 0.00
step_count                    3776.00
stress_level                     5.20
mood_score                       3.20
energy_level                     1.00
caffeine_intake                  2.20
break_minutes                   10.00
Name: 0, dtype: float64


In [8]:
scenario = what_if_prediction(
    record_index=0,
    changes={
        "sleep_hours": 7.5
    }
)

scenario

{'record_index': 0,
 'baseline_prediction': 23.52,
 'scenario_prediction': 41.81,
 'estimated_change': 18.29,
 'changes': {'sleep_hours': 7.5}}

In [9]:
scenario = what_if_prediction(
    record_index=0,
    changes={
        "sleep_hours": 7.5,
        "exercise_minutes": 30,
        "screen_time_hours": 6.0
    }
)

scenario

{'record_index': 0,
 'baseline_prediction': 23.52,
 'scenario_prediction': 42.59,
 'estimated_change': 19.07,
 'changes': {'sleep_hours': 7.5,
  'exercise_minutes': 30,
  'screen_time_hours': 6.0}}

In [10]:
feature_ranges = {
    "sleep_hours": (3, 10.5),
    "sleep_quality": (1, 10),
    "wake_time_consistency": (1, 10),
    "study_work_hours": (0, 14),
    "focus_sessions": (0, 10),
    "screen_time_hours": (1, 16),
    "social_media_hours": (0, 9.99),
    "notification_interruptions": (0, 150),
    "exercise_minutes": (0, 128),
    "step_count": (505, 19944),
    "stress_level": (1, 10),
    "mood_score": (1, 10),
    "energy_level": (1, 10),
    "caffeine_intake": (0, 6),
    "break_minutes": (0, 218)
}

In [11]:
def what_if_prediction(record_index, changes):
    baseline_row = df.loc[record_index].copy()
    scenario_row = baseline_row.copy()

    for feature, value in changes.items():

        if feature not in behavioral_features:
            raise ValueError(
                f"{feature} is not a valid behavioral feature."
            )

        if feature in feature_ranges:
            min_value, max_value = feature_ranges[feature]

            if not min_value <= value <= max_value:
                raise ValueError(
                    f"{feature} must be between "
                    f"{min_value} and {max_value}."
                )

        scenario_row[feature] = value

    baseline_input = prepare_model_input(baseline_row)
    scenario_input = prepare_model_input(scenario_row)

    baseline_prediction = regressor.predict(baseline_input)[0]
    scenario_prediction = regressor.predict(scenario_input)[0]

    change = scenario_prediction - baseline_prediction

    return {
        "record_index": record_index,
        "baseline_prediction": round(float(baseline_prediction), 2),
        "scenario_prediction": round(float(scenario_prediction), 2),
        "estimated_change": round(float(change), 2),
        "changes": changes
    }

In [12]:
scenario = what_if_prediction(
    record_index=0,
    changes={
        "sleep_hours": 7.5,
        "exercise_minutes": 30,
        "screen_time_hours": 6.0
    }
)

scenario

{'record_index': 0,
 'baseline_prediction': 23.52,
 'scenario_prediction': 42.59,
 'estimated_change': 19.07,
 'changes': {'sleep_hours': 7.5,
  'exercise_minutes': 30,
  'screen_time_hours': 6.0}}

In [14]:
try:
    what_if_prediction(
        record_index=0,
        changes={"sleep_hours": 25}
    )
except ValueError as e:
    print("Invalid input:", e)

Invalid input: sleep_hours must be between 3 and 10.5.
